# Cross-Validation Experiments

This notebook extends the CatBoost baseline with **stratified 5-fold cross-validation** to obtain a more robust AUC estimate, then runs controlled experiments that push performance further by increasing iterations and tuning early stopping behaviour.

---

## Table of Contents

1. [Setup and Data Loading](#cv-setup-and-data-loading)
   - 1.1 [Imports](#cv-imports)
   - 1.2 [Load Datasets & Preprocessing](#cv-load-datasets)
2. [Baseline 5-Fold CV](#cv-baseline-cv)
3. [Experiment 01: More Iterations + Early Stopping](#cv-exp01)
4. [Experiment 01B: Probe + Optimal Iterations](#cv-exp01b)
5. [Final Model & Submission](#cv-final-model)
6. [Summary](#cv-summary)

---

<a id="cv-setup-and-data-loading" name="cv-setup-and-data-loading"></a>

## 1. Setup and Data Loading

<a id="cv-imports" name="cv-imports"></a>

### 1.1 Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

<a id="cv-load-datasets" name="cv-load-datasets"></a>

### 1.2 Load Datasets & Preprocessing

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

X = train.drop(columns=["addicted_label", "id"]).copy()
y = train["addicted_label"].copy()

X_test = test.drop(columns=["id"]).copy()

cat_cols = [
    "gender",
    "stress_level",
    "academic_work_impact",
]

for col in cat_cols:
    X[col] = X[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

Train shape: (691369, 14)
Test shape: (296302, 13)


<a id="cv-baseline-cv" name="cv-baseline-cv"></a>

## 2. Baseline 5-Fold CV

Establish a cross-validated baseline using the same CatBoost hyperparameters as the single-split baseline: **500 iterations**, `learning_rate=0.05`, `depth=6`, and **no early stopping**. Stratified 5-fold CV gives a more reliable AUC estimate than a single train/validation split, and the standard deviation across folds helps gauge model stability.

In [3]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,  # shuffle the data before splitting into folds
    random_state=42,
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,            # suppress per-round output
        allow_writing_files=False,  # disable on-disk CatBoost cache
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred,
    )

    fold_scores.append(fold_auc)

    print(f"Fold {fold}: {fold_auc:.6f}")

Fold 1: 0.950670
Fold 2: 0.951268
Fold 3: 0.951656
Fold 4: 0.952284
Fold 5: 0.951368


In [4]:
print(f"CV Mean: {np.mean(fold_scores):.6f}")
print(f"CV Std:  {np.std(fold_scores):.6f}")

CV Mean: 0.951449
CV Std:  0.000527


The 5-fold CV mean of **0.9514** is nearly identical to the single hold-out AUC of **0.9509** from the baseline notebook — a difference of less than 0.001. This agreement confirms that the earlier single-split estimate was not an artefact of a lucky split. The low standard deviation across folds (**0.000527**) further indicates the model is stable and the score generalises consistently across different subsets of the training data.

<a id="cv-exp01" name="cv-exp01"></a>

## 3. Experiment 01: More Iterations + Early Stopping

The baseline CV confirmed the model was undertrained at 500 iterations — validation AUC had not plateaued, and the learning curve showed continued improvement up to the ceiling. This experiment raises the limit to **3,000 iterations** and introduces `early_stopping_rounds=100` to prevent premature termination during the late phase of training, where gains are small but accumulate slowly. All other hyperparameters (`learning_rate=0.05`, `depth=6`) remain unchanged.

Note that in both the baseline and this experiment, best iterations landed at or near the respective ceilings, meaning early stopping never actually triggered the halt — the model was simply cut off by the iteration limit each time. This makes clear that the performance gap between the two runs is attributable to training duration rather than overfitting behaviour, and that further headroom likely still exists beyond 3,000 iterations.

In [5]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

exp1_scores = []
exp1_best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y),start=1):
    
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    best_iteration = model.get_best_iteration() + 1

    exp1_scores.append(fold_auc)
    exp1_best_iterations.append(best_iteration)

    print(
        f"Fold {fold}: "
        f"AUC={fold_auc:.6f}, "
        f"Best iteration={best_iteration}"
    )

Fold 1: AUC=0.962148, Best iteration=3000
Fold 2: AUC=0.962613, Best iteration=3000
Fold 3: AUC=0.962926, Best iteration=3000
Fold 4: AUC=0.963516, Best iteration=3000
Fold 5: AUC=0.962518, Best iteration=2999


In [ ]:
print()
print("Baseline CV Mean: 0.951449")
print(f"Experiment CV Mean: {np.mean(exp1_scores):.6f}")
print(f"Experiment CV Std:  {np.std(exp1_scores):.6f}")

print()
print("Best iterations:", exp1_best_iterations)
print("Mean best iteration:",round(np.mean(exp1_best_iterations)))


Baseline CV Mean: 0.951449
Experiment CV Mean: 0.962744
Experiment CV Std:  0.000459

Best iterations: [3000, 3000, 3000, 3000, 2999]
Mean best iteration: 3000


Compared with the baseline, increasing the training budget from 500 to 3,000 iterations raises the mean CV AUC from **0.951449** to **0.962744**, a gain of about **0.0113**. The fold-to-fold variation also remains very small (**0.000459**), confirming that the improvement is stable rather than driven by one favourable split. With the best iterations clustered at the maximum allowed value, the 3,000-round result should be treated as a strong intermediate benchmark rather than a fully tuned endpoint.

<a id="cv-exp01b" name="cv-exp01b"></a>

## 4. Experiment 01B: Probe + Optimal Iterations

A two-stage search is used to choose a better-supported training budget:

**Step 1 — Single-fold probe:** Run one fold with a high ceiling of **10,000 iterations** and `early_stopping_rounds=100` to find where validation AUC truly saturates. `verbose=500` prints progress every 500 rounds so the learning curve remains visible.

**Step 2 — Full 5-fold CV:** Use the probe's best iteration count as the ceiling for a full cross-validation run with a wider early-stopping window (`early_stopping_rounds=200`) for a more robust estimate.

In [ ]:
skf_probe = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, valid_idx = next(skf_probe.split(X, y))

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

probe_model = CatBoostClassifier(
    iterations=10000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=500,
    allow_writing_files=False,
)

probe_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
)

probe_best_iteration = probe_model.get_best_iteration() + 1

probe_pred = probe_model.predict_proba(X_valid)[:, 1]

probe_auc = roc_auc_score(
    y_valid,
    probe_pred
)

print("Probe AUC:", round(probe_auc, 6))
print("Best iteration:", probe_best_iteration)

0:	test: 0.9043297	best: 0.9043297 (0)	total: 79.4ms	remaining: 13m 13s
500:	test: 0.9506777	best: 0.9506777 (500)	total: 49.5s	remaining: 15m 38s
1000:	test: 0.9567446	best: 0.9567446 (1000)	total: 1m 56s	remaining: 17m 24s
1500:	test: 0.9593642	best: 0.9593642 (1500)	total: 3m 32s	remaining: 20m 2s
2000:	test: 0.9607543	best: 0.9607543 (2000)	total: 4m 44s	remaining: 18m 56s
2500:	test: 0.9616662	best: 0.9616662 (2500)	total: 5m 31s	remaining: 16m 33s
3000:	test: 0.9621501	best: 0.9621501 (3000)	total: 6m 34s	remaining: 15m 21s
3500:	test: 0.9624841	best: 0.9624841 (3500)	total: 8m 18s	remaining: 15m 25s
4000:	test: 0.9627070	best: 0.9627075 (3999)	total: 9m 12s	remaining: 13m 48s
4500:	test: 0.9628853	best: 0.9628863 (4493)	total: 10m 2s	remaining: 12m 16s
5000:	test: 0.9630300	best: 0.9630303 (4991)	total: 11m 11s	remaining: 11m 10s
5500:	test: 0.9631350	best: 0.9631381 (5470)	total: 12m 58s	remaining: 10m 37s
6000:	test: 0.9631822	best: 0.9631822 (6000)	total: 13m 40s	remaining: 9

In [8]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

exp1b_scores = []
exp1b_best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=10000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=200,
    )

    pred = model.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(y_valid, pred)
    best_iter = model.get_best_iteration() + 1

    exp1b_scores.append(auc)
    exp1b_best_iterations.append(best_iter)

    print(
        f"Fold {fold}: "
        f"AUC={auc:.6f}, "
        f"Best iteration={best_iter}"
    )

print()
print(f"CV Mean: {np.mean(exp1b_scores):.6f}")
print(f"CV Std:  {np.std(exp1b_scores):.6f}")
print("Best iterations:", exp1b_best_iterations)
print("Mean best iteration:",round(np.mean(exp1b_best_iterations)))

Fold 1: AUC=0.963389, Best iteration=9145
Fold 2: AUC=0.963901, Best iteration=8625
Fold 3: AUC=0.964157, Best iteration=7514
Fold 4: AUC=0.964813, Best iteration=7780
Fold 5: AUC=0.963776, Best iteration=8050

CV Mean: 0.964007
CV Std:  0.000473
Best iterations: [9145, 8625, 7514, 7780, 8050]
Mean best iteration: 8223


<a id="cv-final-model" name="cv-final-model"></a>

## 5. Final Model & Submission

Experiment 01B's full 5-fold CV converged at a mean best iteration of **8,223**. The final model is retrained on the complete training set `(X, y)` using that fixed iteration count — no `eval_set` or early stopping, since the ceiling has already been determined. Predictions are then generated for the test set and written to a submission CSV.

In [9]:
final_model_v2 = CatBoostClassifier(
    iterations=8223,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=500,
    allow_writing_files=False,
)

final_model_v2.fit(
    X,
    y,
    cat_features=cat_cols,
)

0:	total: 89.8ms	remaining: 12m 18s
500:	total: 47.7s	remaining: 12m 15s
1000:	total: 1m 35s	remaining: 11m 30s
1500:	total: 2m 23s	remaining: 10m 42s
2000:	total: 3m 11s	remaining: 9m 54s
2500:	total: 3m 58s	remaining: 9m 6s
3000:	total: 4m 46s	remaining: 8m 19s
3500:	total: 5m 34s	remaining: 7m 31s
4000:	total: 6m 22s	remaining: 6m 44s
4500:	total: 7m 11s	remaining: 5m 56s
5000:	total: 7m 59s	remaining: 5m 8s
5500:	total: 8m 47s	remaining: 4m 21s
6000:	total: 9m 35s	remaining: 3m 33s
6500:	total: 10m 23s	remaining: 2m 45s
7000:	total: 11m 10s	remaining: 1m 57s
7500:	total: 11m 58s	remaining: 1m 9s
8000:	total: 12m 45s	remaining: 21.2s
8222:	total: 13m 6s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=8223, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=500)

In [10]:
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

test_pred_v2 = final_model_v2.predict_proba(X_test)[:, 1]

submission_v2 = sample_submission.copy()
submission_v2["addicted_label"] = test_pred_v2

display(submission_v2.head())
print(submission_v2.shape)
print(submission_v2.isna().sum())

,id,addicted_label
0,691369,0.999529
1,691370,0.967484
2,691371,0.940303
3,691372,0.990979
4,691373,0.998337


(296302, 2)
id                0
addicted_label    0
dtype: int64


In [11]:
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

submission_path = (
    SUBMISSION_DIR /
    "catboost_v2_8223_iterations.csv"
)

submission_v2.to_csv(
    submission_path,
    index=False,
)

print("Saved to:", submission_path)

Saved to: /Users/c.c./Developer/Competitions/predicting-smartphone-addiction/submissions/catboost_v2_8223_iterations.csv


<a id="cv-summary" name="cv-summary"></a>

## 6. Summary

| Run | Training setup | CV AUC (mean) | CV AUC (std) | Public LB |
|---|---|---:|---:|---:|
| Baseline | 500 iterations | 0.951449 | 0.000527 | 0.95227 |
| Exp 01 | 3,000 iterations, ESR 100 | 0.962744 | 0.000459 | — |
| Exp 01B / Final | 10,000 ceiling, ESR 200 → 8,223 final iterations | **0.964007** | 0.000473 | **0.96538** |

### Key findings

- The baseline produced a stable reference point: its CV AUC (**0.951449**) closely matched the first submission's Public LB score (**0.95227**), supporting the reliability of the cross-validation setup.
- Increasing the training budget from 500 to 3,000 iterations delivered the largest gain (**+0.011295 CV AUC**). Because the best iteration stayed at the ceiling across folds, the model was still undertrained.
- The high-ceiling probe and full CV run placed the useful convergence range at roughly 7,500–9,100 iterations. Using the mean best iteration (**8,223**) added a further **+0.001263 CV AUC** over Exp 01, showing smaller but consistent returns from longer training.
- Retraining on all available training data for 8,223 iterations achieved a Public LB score of **0.96538**, an improvement of **+0.01311** over the baseline submission. Its close agreement with the CV estimate (**0.964007**) also suggests that the selected training budget generalises well.

![CatBoost v2 submission with 8,223 iterations — Public LB 0.96538](../images/catboost_v2_8223_public_lb.png)

*Submission result for `catboost_v2_8223_iterations.csv`.*

### Next direction

With the training budget now well calibrated, the next experiment should shift from additional iteration tuning to feature engineering—especially interaction and ratio features derived from the screen-time variables highlighted by the baseline feature importance.